In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# Ensure value columns are numeric where applicable
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
log['value'] = pd.to_numeric(log['value'], errors='coerce')

# Aggregation helper: count, min(timestamp), max(timestamp), median(value), p95(value)
def summarize_series(df, group_cols, value_col='value'):
    g = df.groupby(group_cols)
    summary = g.agg(
        count=(value_col, 'count'),
        min_ts=('timestamp', 'min'),
        max_ts=('timestamp', 'max'),
        median=(value_col, 'median'),
        p95=(value_col, lambda x: x.dropna().quantile(0.95))
    ).reset_index()
    summary = summary.sort_values('p95', ascending=False).head(20)
    return summary

# 1) metric summary
metric_summary = summarize_series(metric, ['cmdb_id', 'kpi_name'])

# 2) trace summary
trace_summary = summarize_series(trace, ['cmdb_id', 'trace_name'])

# 3) log summary
log_summary = summarize_series(log, ['cmdb_id', 'log_name'])

# 4) error_logs summary: total row count and up to first 20 distinct (cmdb_id, min_ts, max_ts) groups
error_count = error_logs.shape[0]
error_groups = error_logs.groupby('cmdb_id').agg(
    count=('timestamp', 'count'),
    min_ts=('timestamp', 'min'),
    max_ts=('timestamp', 'max')
).reset_index().head(20)

# Display compact aggregated results (variables separated by commas for IPython)
metric_summary, trace_summary, log_summary, error_count, error_groups

```
Out[1]:
```
summary = (
    "Summary of telemetry (global / all-time):\n"
    "- Metrics: top P95 values\n"
    "  1) payment - diskio: P95 ≈ 7.61e+09 (count=13, range 2024-01-19 06:57 -> 07:09 UTC)\n"
    "  2) catalogue-db - mem: P95 ≈ 4.32472e+08 (count=25, range 2024-01-19 06:45 -> 07:09 UTC)\n"
    "  3) orders - mem: P95 ≈ 3.28122e+08 (count=25, range 2024-01-19 06:45 -> 07:09 UTC)\n"
    "  4) payment - mem: P95 ≈ 3.14505e+08 (count=25, range 2024-01-19 06:45 -> 07:09 UTC)\n"
    "  5) shipping, queue-master, user-db and others also show notable mem or diskio P95s in the 1e6–1e8 range.\n"
    "- Traces: no trace groups present (trace summary is empty).\n"
    "- Logs: highest log.row_count P95s\n"
    "  • front-end log.row_count: P95 ≈ 1829.85 (count=24, 06:45 -> 07:08 UTC)\n"
    "  • orders log.row_count: P95 ≈ 713.30 (count=24)\n"
    "  • user and carts-db also show elevated row_count P95s (~559 and ~552 respectively).\n"
    "- Error logs: zero rows in error_logs.csv (no error log groups to report).\n\n"
    "Overall: metric telemetry highlights very high disk I/O for payment and elevated memory usage across several services (catalogue-db, orders, payment, shipping). No trace data and no recorded error logs; front-end and orders show the highest log volume. These global summaries can be used as baselines before any incident-window filtering."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id kpi_name  count                    min_ts                    max_ts        median           p95
42            payment   diskio     13 2024-01-19 06:57:00+00:00 2024-01-19 07:09:00+00:00  7.403288e+09  7.609728e+09
21       catalogue-db      mem     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  4.324365e+08  4.324720e+08
34             orders      mem     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  3.276534e+08  3.281215e+08
46            payment      mem     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  2.298123e+08  3.145051e+08
67           shipping      mem     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  3.065018e+08  3.065057e+08
51       queue-master      mem     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  2.684549e+08  2.684633e+08
79            user-db      mem     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  2.335189e+08  2.360370e+08
4               carts      mem     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  2.225482e+08  2.226037e+08
9            carts-db      mem     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  1.363674e+08  1.387620e+08
39          orders-db      mem     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  1.275538e+08  1.286558e+08
27          front-end      mem     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  1.043279e+08  1.090449e+08
55           rabbitmq      mem     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  9.845255e+07  9.845531e+07
74               user      mem     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  8.881220e+06  9.035134e+06
16          catalogue      mem     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  6.443691e+06  6.509022e+06
58  rabbitmq-exporter      mem     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  4.575232e+06  4.575232e+06
8            carts-db   diskio     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  3.185060e+06  3.645584e+06
78            user-db   diskio     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  2.767457e+06  2.963810e+06
62         session-db      mem     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  2.909252e+06  2.963743e+06
38          orders-db   diskio     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  7.542393e+05  8.509440e+05
50       queue-master   diskio     25 2024-01-19 06:45:00+00:00 2024-01-19 07:09:00+00:00  2.844327e+04  4.181995e+04, Empty DataFrame
Columns: [cmdb_id, trace_name, count, min_ts, max_ts, median, p95]
Index: [],          cmdb_id         log_name  count                    min_ts                    max_ts  median      p95
7      front-end    log.row_count     24 2024-01-19 06:45:00+00:00 2024-01-19 07:08:00+00:00  1720.5  1829.85
9         orders    log.row_count     24 2024-01-19 06:45:00+00:00 2024-01-19 07:08:00+00:00   109.0   713.30
19          user    log.row_count     24 2024-01-19 06:45:00+00:00 2024-01-19 07:08:00+00:00   522.5   559.20
3       carts-db    log.row_count     24 2024-01-19 06:45:00+00:00 2024-01-19 07:08:00+00:00   474.5   552.15
13  queue-master    log.row_count     23 2024-01-19 06:46:00+00:00 2024-01-19 07:08:00+00:00   477.0   537.30
5      catalogue    log.row_count     24 2024-01-19 06:45:00+00:00 2024-01-19 07:08:00+00:00   148.0   155.55
1          carts    log.row_count     23 2024-01-19 06:45:00+00:00 2024-01-19 07:08:00+00:00    75.0   122.50
11       payment    log.row_count     23 2024-01-19 06:46:00+00:00 2024-01-19 07:08:00+00:00    93.0    99.70
17      shipping    log.row_count     23 2024-01-19 06:46:00+00:00 2024-01-19 07:08:00+00:00    53.0    59.70
15    session-db    log.row_count      5 2024-01-19 06:47:00+00:00 2024-01-19 07:07:00+00:00     5.0     5.00
2       carts-db  log.error_count     24 2024-01-19 06:45:00+00:00 2024-01-19 07:08:00+00:00     0.0     0.00
0          carts  log.error_count     23 2024-01-19 06:45:00+00:00 2024-01-19 07:08:00+00:00     0.0     0.00
10       payment  log.error_count     23 2024-01-19 06:46:00+00:00 2024-01-19 07:08:00+00:00     0.0     0.00
8         orders  log.error_count     24 2024-01-19 06:45:00+00:00 2024-01-19 07:08:00+00:00     0.0     0.00
4      catalogue  log.error_count     24 2024-01-19 06:45:00+00:00 2024-01-19 07:08:00+00:00     0.0     0.00
6      front-end  log.error_count     24 2024-01-19 06:45:00+00:00 2024-01-19 07:08:00+00:00     0.0     0.00
14    session-db  log.error_count      5 2024-01-19 06:47:00+00:00 2024-01-19 07:07:00+00:00     0.0     0.00
12  queue-master  log.error_count     23 2024-01-19 06:46:00+00:00 2024-01-19 07:08:00+00:00     0.0     0.00
16      shipping  log.error_count     23 2024-01-19 06:46:00+00:00 2024-01-19 07:08:00+00:00     0.0     0.00
18          user  log.error_count     24 2024-01-19 06:45:00+00:00 2024-01-19 07:08:00+00:00     0.0     0.00, 0, Empty DataFrame
Columns: [cmdb_id, count, min_ts, max_ts]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse previously loaded dataframes if present; otherwise load from CSVs
try:
    metric
    trace
    log
    error_logs
except NameError:
    metric = pd.read_csv("metric.csv")
    trace = pd.read_csv("trace.csv")
    log = pd.read_csv("log.csv")
    error_logs = pd.read_csv("error_logs.csv")

# Ensure timestamps parsed as UTC datetimes and values numeric
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
log['value'] = pd.to_numeric(log['value'], errors='coerce')

# Incident window (UTC)
start = pd.Timestamp("2024-01-19 06:42:00", tz="UTC")
end   = pd.Timestamp("2024-01-19 07:12:00", tz="UTC")

# ===== 1) Metrics anomalies =====
# 1a: compute global P95 per (cmdb_id, kpi_name) using full series (global thresholds)
metric_p95 = metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value'] \
                   .quantile(0.95).reset_index(name='global_p95')

# 1b: filter metric series to incident window
metric_win = metric[(metric['timestamp'] >= start) & (metric['timestamp'] <= end)].copy()

if not metric_win.empty:
    # merge global p95 into window rows
    metric_win = metric_win.merge(metric_p95, on=['cmdb_id', 'kpi_name'], how='left')

    # boolean whether row exceeds global p95
    metric_win['gt_p95'] = metric_win['value'] > metric_win['global_p95']

    # aggregate per group within window
    metric_agg = metric_win.groupby(['cmdb_id', 'kpi_name'], dropna=False).agg(
        total_points_in_window = ('value', 'count'),
        count_points_gt_p95 = ('gt_p95', 'sum'),
        earliest_timestamp_gt_p95 = ('timestamp', lambda s: s[metric_win.loc[s.index, 'gt_p95']].min() if s.size>0 else pd.NaT),
        max_value_in_window = ('value', 'max'),
    ).reset_index()

    # attach global_p95
    metric_agg = metric_agg.merge(metric_p95, on=['cmdb_id', 'kpi_name'], how='left')

    # severity_ratio (handle division by zero)
    metric_agg['severity_ratio'] = metric_agg['max_value_in_window'] / metric_agg['global_p95']
    metric_agg.loc[metric_agg['global_p95'] == 0, 'severity_ratio'] = np.inf

    # filter rows where count_points_gt_p95 > 0
    metric_anoms = metric_agg[metric_agg['count_points_gt_p95'] > 0].copy()

    # sort and limit
    metric_anoms = metric_anoms.sort_values(['count_points_gt_p95', 'severity_ratio'], ascending=[False, False]) \
                               [['cmdb_id','kpi_name','global_p95','total_points_in_window',
                                 'count_points_gt_p95','earliest_timestamp_gt_p95','max_value_in_window','severity_ratio']] \
                               .head(20)
else:
    metric_anoms = pd.DataFrame(columns=['cmdb_id','kpi_name','global_p95','total_points_in_window',
                                         'count_points_gt_p95','earliest_timestamp_gt_p95','max_value_in_window','severity_ratio'])

# ===== 2) Logs anomalies =====
log_p95 = log.groupby(['cmdb_id', 'log_name'], dropna=False)['value'] \
             .quantile(0.95).reset_index(name='global_p95')

log_win = log[(log['timestamp'] >= start) & (log['timestamp'] <= end)].copy()

if not log_win.empty:
    log_win = log_win.merge(log_p95, on=['cmdb_id', 'log_name'], how='left')
    log_win['gt_p95'] = log_win['value'] > log_win['global_p95']

    log_agg = log_win.groupby(['cmdb_id','log_name'], dropna=False).agg(
        total_points_in_window = ('value','count'),
        count_points_gt_p95 = ('gt_p95','sum'),
        earliest_timestamp_gt_p95 = ('timestamp', lambda s: s[log_win.loc[s.index, 'gt_p95']].min() if s.size>0 else pd.NaT),
        max_value_in_window = ('value','max'),
    ).reset_index()

    log_agg = log_agg.merge(log_p95, on=['cmdb_id','log_name'], how='left')
    log_agg['severity_ratio'] = log_agg['max_value_in_window'] / log_agg['global_p95']
    log_agg.loc[log_agg['global_p95'] == 0, 'severity_ratio'] = np.inf

    log_anoms = log_agg[log_agg['count_points_gt_p95'] > 0].copy()
    log_anoms = log_anoms.sort_values(['count_points_gt_p95','severity_ratio'], ascending=[False, False]) \
                         [['cmdb_id','log_name','global_p95','total_points_in_window',
                           'count_points_gt_p95','earliest_timestamp_gt_p95','max_value_in_window','severity_ratio']] \
                         .head(20)
else:
    log_anoms = pd.DataFrame(columns=['cmdb_id','log_name','global_p95','total_points_in_window',
                                      'count_points_gt_p95','earliest_timestamp_gt_p95','max_value_in_window','severity_ratio'])

# ===== 3) Error logs in-window summary =====
error_win = error_logs[(error_logs['timestamp'] >= start) & (error_logs['timestamp'] <= end)].copy()
error_count_in_window = int(error_win.shape[0])

if not error_win.empty:
    error_groups_in_window = error_win.groupby('cmdb_id', dropna=False).agg(
        count_in_window = ('timestamp','count'),
        min_ts = ('timestamp','min'),
        max_ts = ('timestamp','max')
    ).reset_index().head(20)
else:
    error_groups_in_window = pd.DataFrame(columns=['cmdb_id','count_in_window','min_ts','max_ts'])

# ===== 4) Traces anomalies =====
trace_anoms = pd.DataFrame(columns=['cmdb_id','trace_name','global_p95','total_points_in_window',
                                    'count_points_gt_p95','earliest_timestamp_gt_p95','max_value_in_window','severity_ratio'])
if not trace.empty:
    trace_p95 = trace.groupby(['cmdb_id','trace_name'], dropna=False)['value'] \
                     .quantile(0.95).reset_index(name='global_p95')
    trace_win = trace[(trace['timestamp'] >= start) & (trace['timestamp'] <= end)].copy()
    if not trace_win.empty:
        trace_win = trace_win.merge(trace_p95, on=['cmdb_id','trace_name'], how='left')
        trace_win['gt_p95'] = trace_win['value'] > trace_win['global_p95']
        trace_agg = trace_win.groupby(['cmdb_id','trace_name'], dropna=False).agg(
            total_points_in_window = ('value','count'),
            count_points_gt_p95 = ('gt_p95','sum'),
            earliest_timestamp_gt_p95 = ('timestamp', lambda s: s[trace_win.loc[s.index, 'gt_p95']].min() if s.size>0 else pd.NaT),
            max_value_in_window = ('value','max'),
        ).reset_index()
        trace_agg = trace_agg.merge(trace_p95, on=['cmdb_id','trace_name'], how='left')
        trace_agg['severity_ratio'] = trace_agg['max_value_in_window'] / trace_agg['global_p95']
        trace_agg.loc[trace_agg['global_p95'] == 0, 'severity_ratio'] = np.inf
        trace_anoms = trace_agg[trace_agg['count_points_gt_p95'] > 0].copy()
        trace_anoms = trace_anoms.sort_values(['count_points_gt_p95','severity_ratio'], ascending=[False, False]) \
                                 [['cmdb_id','trace_name','global_p95','total_points_in_window',
                                   'count_points_gt_p95','earliest_timestamp_gt_p95','max_value_in_window','severity_ratio']] \
                                 .head(20)

# Final compact outputs (variables displayed for IPython)
metric_anoms, log_anoms, trace_anoms, error_count_in_window, error_groups_in_window

```
Out[2]:
```
Incident window: 2024-01-19 06:42:00 → 2024-01-19 07:12:00 UTC.

Summary of findings (compact):

1) Metrics anomalies (groups with values > global P95 inside the window)
- queue-master.diskio — strongest signal: 2 points > global P95, max ≈ 92,407 vs global P95 ≈ 41,820 (severity ≈ 2.21). (earliest breach 2024-01-19 06:55 UTC)
- orders.cpu — 2 points > P95, max ≈ 3.70 vs P95 ≈ 1.96 (severity ≈ 1.89). (earliest 06:45)
- shipping.cpu — 2 points > P95, max ≈ 0.603 vs P95 ≈ 0.350 (severity ≈ 1.72).
- front-end.error and orders.error — each 2 points > P95, severity ≈ 1.49 and 1.48 respectively.
- rabbitmq-exporter.cpu, session-db.diskio, orders latency-50, catalogue.workload, orders.socket, carts.cpu, etc. — each shows 2 points > P95 with modest severity (mostly 1.03–1.32).
- Most metric anomaly groups have only 2 points above global P95 (i.e., short spikes), though queue-master.diskio and orders.cpu show the largest severity.

2) Log anomalies
- orders.log.row_count — 2 points > P95; max 1560 vs P95 713 (severity ≈ 2.19). (earliest breach 06:45)
- user.log.row_count — 2 points > P95; max 585 vs P95 559 (severity ≈ 1.05).
- front-end.log.row_count — 2 points > P95; max 1885 vs P95 1829 (severity ≈ 1.03).
- catalogue, carts-db, carts, queue-master, shipping, payment also have 2-point row_count spikes above their global P95s (small to modest severity).

3) Traces
- No trace-group anomalies found (no groups with values above global P95 in the window).

4) Error logs
- No error_log rows found inside the incident window (count = 0).

Brief interpretation / next steps
- The strongest metric outlier is high disk I/O on queue-master (severity ≈ 2.2) and elevated CPU in orders — these are the highest-priority signals to inspect.
- Front-end and orders show elevated log volumes and front-end/errors also show metric-level error spikes — investigate front-end and orders for request/error bursts.
- Check queue-master disk subsystem and orders service processes (CPU, threads, request patterns). Session-db and user-db show smaller disk I/O elevations worth reviewing.
- No trace data and no error_log entries are available in the window, so rely on the above metric/log spikes and collect traces / application logs if deeper diagnosis is needed.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name    global_p95  total_points_in_window  count_points_gt_p95 earliest_timestamp_gt_p95  max_value_in_window  severity_ratio
50       queue-master      diskio  4.181995e+04                      25                    2 2024-01-19 06:55:00+00:00         9.244078e+04        2.210447
30             orders         cpu  1.955887e+00                      25                    2 2024-01-19 06:45:00+00:00         3.701913e+00        1.892703
64           shipping         cpu  3.503103e-01                      25                    2 2024-01-19 06:50:00+00:00         6.032024e-01        1.721909
24          front-end       error  1.136200e+00                      25                    2 2024-01-19 06:45:00+00:00         1.691233e+00        1.488500
31             orders       error  1.136133e+00                      25                    2 2024-01-19 06:45:00+00:00         1.679150e+00        1.477952
57  rabbitmq-exporter         cpu  1.201886e-04                      25                    2 2024-01-19 07:01:00+00:00         1.588227e-04        1.321445
61         session-db      diskio  1.158932e+04                      25                    2 2024-01-19 06:47:00+00:00         1.434663e+04        1.237918
32             orders  latency-50  8.590186e-01                      25                    2 2024-01-19 06:58:00+00:00         9.682155e-01        1.127118
18          catalogue    workload  3.899977e+00                      25                    2 2024-01-19 06:46:00+00:00         4.333000e+00        1.111032
35             orders      socket  2.438333e+01                      25                    2 2024-01-19 06:58:00+00:00         2.683333e+01        1.100478
0               carts         cpu  1.424838e+00                      25                    2 2024-01-19 06:46:00+00:00         1.534840e+00        1.077203
49       queue-master         cpu  2.851466e-01                      25                    2 2024-01-19 06:54:00+00:00         3.043435e-01        1.067323
44            payment  latency-50  5.648092e-03                      24                    2 2024-01-19 07:00:00+00:00         5.955068e-03        1.054350
28          front-end      socket  3.500000e+01                      25                    2 2024-01-19 06:58:00+00:00         3.686667e+01        1.053333
38          orders-db      diskio  8.509440e+05                      25                    2 2024-01-19 06:49:00+00:00         8.953509e+05        1.052185
36             orders    workload  1.980417e+00                      25                    2 2024-01-19 06:46:00+00:00         2.067767e+00        1.044107
78            user-db      diskio  2.963810e+06                      25                    2 2024-01-19 06:53:00+00:00         3.089072e+06        1.042264
53           rabbitmq         cpu  8.067987e-01                      25                    2 2024-01-19 06:54:00+00:00         8.392360e-01        1.040205
33             orders  latency-90  2.180261e+00                      25                    2 2024-01-19 07:04:00+00:00         2.263347e+00        1.038108
6               carts    workload  7.671293e+00                      25                    2 2024-01-19 06:46:00+00:00         7.963550e+00        1.038097,          cmdb_id       log_name  global_p95  total_points_in_window  count_points_gt_p95 earliest_timestamp_gt_p95  max_value_in_window  severity_ratio
9         orders  log.row_count      713.30                      24                    2 2024-01-19 06:45:00+00:00                 1560        2.187018
19          user  log.row_count      559.20                      24                    2 2024-01-19 06:46:00+00:00                  585        1.046137
7      front-end  log.row_count     1829.85                      24                    2 2024-01-19 06:46:00+00:00                 1885        1.030139
5      catalogue  log.row_count      155.55                      24                    2 2024-01-19 06:46:00+00:00                  160        1.028608
3       carts-db  log.row_count      552.15                      24                    2 2024-01-19 07:01:00+00:00                  565        1.023273
1          carts  log.row_count      122.50                      23                    2 2024-01-19 06:48:00+00:00                  125        1.020408
13  queue-master  log.row_count      537.30                      23                    2 2024-01-19 06:50:00+00:00                  540        1.005025
17      shipping  log.row_count       59.70                      23                    2 2024-01-19 06:50:00+00:00                   60        1.005025
11       payment  log.row_count       99.70                      23                    2 2024-01-19 06:50:00+00:00                  100        1.003009, Empty DataFrame
Columns: [cmdb_id, trace_name, global_p95, total_points_in_window, count_points_gt_p95, earliest_timestamp_gt_p95, max_value_in_window, severity_ratio]
Index: [], 0, Empty DataFrame
Columns: [cmdb_id, count_in_window, min_ts, max_ts]
Index: [])```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse loaded dataframes if available, otherwise load
try:
    metric
    log
except NameError:
    metric = pd.read_csv("metric.csv")
    log = pd.read_csv("log.csv")

# Parse timestamps and ensure numeric values (idempotent if already done)
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
log['value'] = pd.to_numeric(log['value'], errors='coerce')

# Incident window
start = pd.Timestamp("2024-01-19 06:42:00", tz="UTC")
end   = pd.Timestamp("2024-01-19 07:12:00", tz="UTC")

# Compute global P95 thresholds from full series (rule: calculate before filtering)
metric_p95 = metric.groupby(['cmdb_id','kpi_name'], dropna=False)['value'].quantile(0.95).reset_index(name='global_p95')
log_p95 = log.groupby(['cmdb_id','log_name'], dropna=False)['value'].quantile(0.95).reset_index(name='global_p95')

# Requested metric KPI pairs
metric_pairs = [
    ("queue-master", "diskio"),
    ("orders", "cpu"),
    ("shipping", "cpu"),
    ("front-end", "error"),
    ("orders", "error"),
    ("orders", "latency-50"),
    ("session-db", "diskio"),
    ("orders-db", "diskio"),
    ("user-db", "diskio"),
]

# Requested log groups
log_pairs = [
    ("orders", "log.row_count"),
    ("front-end", "log.row_count"),
]

# Helper to extract per-pair series (metrics)
metric_series_list = []
metric_summary_rows = []
for cmdb_id, kpi in metric_pairs:
    # lookup global p95
    row = metric_p95[(metric_p95['cmdb_id']==cmdb_id) & (metric_p95['kpi_name']==kpi)]
    global_p95 = float(row['global_p95'].iloc[0]) if not row.empty else np.nan

    # filter to window and pair
    df_win = metric[
        (metric['timestamp'] >= start) &
        (metric['timestamp'] <= end) &
        (metric['cmdb_id'] == cmdb_id) &
        (metric['kpi_name'] == kpi)
    ].copy()

    df_win = df_win.sort_values('timestamp').head(40)
    # ensure requested output columns and name
    if not df_win.empty:
        df_out = df_win.loc[:, ['cmdb_id','kpi_name','timestamp','value']].copy()
    else:
        df_out = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp','value'])
    # attach series identifier for clarity (keeps outputs compact as concatenated)
    df_out.insert(0, 'series', f"{cmdb_id} | {kpi}")
    metric_series_list.append(df_out)

    # summary: total points and timestamps where value > global_p95
    total_points = int(df_win.shape[0])
    if not np.isnan(global_p95) and total_points>0:
        exceed_ts = df_win[df_win['value'] > global_p95]['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ').tolist()
    else:
        exceed_ts = []
    # compact string of timestamps
    exceed_ts_str = ", ".join(exceed_ts) if exceed_ts else ""
    metric_summary_rows.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi,
        'global_p95': global_p95,
        'total_points_in_window': total_points,
        'timestamps_gt_global_p95': exceed_ts_str
    })

# Concatenate metric series (each series limited to first 40 rows)
metric_series_df = pd.concat(metric_series_list, ignore_index=True) if metric_series_list else pd.DataFrame(columns=['series','cmdb_id','kpi_name','timestamp','value'])
metric_summaries_df = pd.DataFrame(metric_summary_rows)

# Helper to extract per-pair series (logs)
log_series_list = []
log_summary_rows = []
for cmdb_id, lname in log_pairs:
    row = log_p95[(log_p95['cmdb_id']==cmdb_id) & (log_p95['log_name']==lname)]
    global_p95 = float(row['global_p95'].iloc[0]) if not row.empty else np.nan

    df_win = log[
        (log['timestamp'] >= start) &
        (log['timestamp'] <= end) &
        (log['cmdb_id'] == cmdb_id) &
        (log['log_name'] == lname)
    ].copy()

    df_win = df_win.sort_values('timestamp').head(40)
    if not df_win.empty:
        df_out = df_win.loc[:, ['cmdb_id','log_name','timestamp','value']].copy()
    else:
        df_out = pd.DataFrame(columns=['cmdb_id','log_name','timestamp','value'])
    df_out.insert(0, 'series', f"{cmdb_id} | {lname}")
    log_series_list.append(df_out)

    total_points = int(df_win.shape[0])
    if not np.isnan(global_p95) and total_points>0:
        exceed_ts = df_win[df_win['value'] > global_p95]['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ').tolist()
    else:
        exceed_ts = []
    exceed_ts_str = ", ".join(exceed_ts) if exceed_ts else ""
    log_summary_rows.append({
        'cmdb_id': cmdb_id,
        'log_name': lname,
        'global_p95': global_p95,
        'total_points_in_window': total_points,
        'timestamps_gt_global_p95': exceed_ts_str
    })

log_series_df = pd.concat(log_series_list, ignore_index=True) if log_series_list else pd.DataFrame(columns=['series','cmdb_id','log_name','timestamp','value'])
log_summaries_df = pd.DataFrame(log_summary_rows)

# Display compact outputs (DataFrames). Variable names are used for IPython output.
metric_series_df, metric_summaries_df, log_series_df, log_summaries_df

```
Out[3]:
```
Incident window: 2024-01-19 06:42:00 → 2024-01-19 07:12:00 UTC.

Metric series (one-line summaries: total_points_in_window; timestamps where value > global P95)
- queue-master | diskio: 25 points; breaches at 2024-01-19T06:55:00Z, 2024-01-19T06:56:00Z
- orders | cpu: 25 points; breaches at 2024-01-19T06:45:00Z, 2024-01-19T07:09:00Z
- shipping | cpu: 25 points; breaches at 2024-01-19T06:50:00Z, 2024-01-19T06:51:00Z
- front-end | error: 25 points; breaches at 2024-01-19T06:45:00Z, 2024-01-19T06:46:00Z
- orders | error: 25 points; breaches at 2024-01-19T06:45:00Z, 2024-01-19T06:46:00Z
- orders | latency-50: 25 points; breaches at 2024-01-19T06:58:00Z, 2024-01-19T07:04:00Z
- session-db | diskio: 25 points; breaches at 2024-01-19T06:47:00Z, 2024-01-19T06:52:00Z
- orders-db | diskio: 25 points; breaches at 2024-01-19T06:49:00Z, 2024-01-19T06:52:00Z
- user-db | diskio: 25 points; breaches at 2024-01-19T06:53:00Z, 2024-01-19T06:54:00Z

Log series (one-line summaries)
- orders | log.row_count: 24 points; breaches at 2024-01-19T06:45:00Z, 2024-01-19T06:46:00Z
- front-end | log.row_count: 24 points; breaches at 2024-01-19T06:46:00Z, 2024-01-19T06:50:00Z

If you want the raw filtered rows for any specific series (up to 40 rows) displayed again, tell me which series to show.

The original code execution output of IPython Kernel is also provided below for reference:

(                    series       cmdb_id kpi_name                 timestamp         value
0    queue-master | diskio  queue-master   diskio 2024-01-19 06:45:00+00:00  1.347053e+04
1    queue-master | diskio  queue-master   diskio 2024-01-19 06:46:00+00:00  1.753184e+04
2    queue-master | diskio  queue-master   diskio 2024-01-19 06:47:00+00:00  3.181478e+04
3    queue-master | diskio  queue-master   diskio 2024-01-19 06:48:00+00:00  3.477656e+04
4    queue-master | diskio  queue-master   diskio 2024-01-19 06:49:00+00:00  3.254736e+04
..                     ...           ...      ...                       ...           ...
220       user-db | diskio       user-db   diskio 2024-01-19 07:05:00+00:00  2.600976e+06
221       user-db | diskio       user-db   diskio 2024-01-19 07:06:00+00:00  2.597367e+06
222       user-db | diskio       user-db   diskio 2024-01-19 07:07:00+00:00  2.949032e+06
223       user-db | diskio       user-db   diskio 2024-01-19 07:08:00+00:00  2.825425e+06
224       user-db | diskio       user-db   diskio 2024-01-19 07:09:00+00:00  1.040596e+06

[225 rows x 5 columns],         cmdb_id    kpi_name    global_p95  total_points_in_window                    timestamps_gt_global_p95
0  queue-master      diskio  4.181995e+04                      25  2024-01-19T06:55:00Z, 2024-01-19T06:56:00Z
1        orders         cpu  1.955887e+00                      25  2024-01-19T06:45:00Z, 2024-01-19T07:09:00Z
2      shipping         cpu  3.503103e-01                      25  2024-01-19T06:50:00Z, 2024-01-19T06:51:00Z
3     front-end       error  1.136200e+00                      25  2024-01-19T06:45:00Z, 2024-01-19T06:46:00Z
4        orders       error  1.136133e+00                      25  2024-01-19T06:45:00Z, 2024-01-19T06:46:00Z
5        orders  latency-50  8.590186e-01                      25  2024-01-19T06:58:00Z, 2024-01-19T07:04:00Z
6    session-db      diskio  1.158932e+04                      25  2024-01-19T06:47:00Z, 2024-01-19T06:52:00Z
7     orders-db      diskio  8.509440e+05                      25  2024-01-19T06:49:00Z, 2024-01-19T06:52:00Z
8       user-db      diskio  2.963810e+06                      25  2024-01-19T06:53:00Z, 2024-01-19T06:54:00Z,                        series    cmdb_id       log_name                 timestamp  value
0      orders | log.row_count     orders  log.row_count 2024-01-19 06:45:00+00:00   1560
1      orders | log.row_count     orders  log.row_count 2024-01-19 06:46:00+00:00    818
2      orders | log.row_count     orders  log.row_count 2024-01-19 06:47:00+00:00    112
3      orders | log.row_count     orders  log.row_count 2024-01-19 06:48:00+00:00    112
4      orders | log.row_count     orders  log.row_count 2024-01-19 06:49:00+00:00    106
..                        ...        ...            ...                       ...    ...
43  front-end | log.row_count  front-end  log.row_count 2024-01-19 07:04:00+00:00   1718
44  front-end | log.row_count  front-end  log.row_count 2024-01-19 07:05:00+00:00   1683
45  front-end | log.row_count  front-end  log.row_count 2024-01-19 07:06:00+00:00   1695
46  front-end | log.row_count  front-end  log.row_count 2024-01-19 07:07:00+00:00   1723
47  front-end | log.row_count  front-end  log.row_count 2024-01-19 07:08:00+00:00   1687

[48 rows x 5 columns],      cmdb_id       log_name  global_p95  total_points_in_window                    timestamps_gt_global_p95
0     orders  log.row_count      713.30                      24  2024-01-19T06:45:00Z, 2024-01-19T06:46:00Z
1  front-end  log.row_count     1829.85                      24  2024-01-19T06:46:00Z, 2024-01-19T06:50:00Z)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```